In [1]:
import pyfastx
import gffutils
import os.path
import numpy as np
import numpy as np
import torch
from torch.utils.data import Dataset
import pickle
import pandas as pd
from scipy.sparse import load_npz
from glob import glob
from math import ceil
import h5py
from src.dataloader import ceil_div, DataPointFull, spliceDataset


In [21]:
data_dir = '../Data/Mouse/v87/'
fasta_file_path = '../Data/Mouse/v87/mm10.fa'
gtf_file_path = '../Data/Mouse/v87/Mus_musculus.GRCm38.87.gtf'

In [22]:
fasta = pyfastx.Fasta(fasta_file_path)
fname = data_dir+gtf_file_path.split('/')[-1][:-4]+'.db'
if not os.path.isfile(fname): 
    gffutils.create_db(gtf_file_path, fname, force=True, disable_infer_genes=True, disable_infer_transcripts=True)

gtf = gffutils.FeatureDB(fname)

In [26]:
genes = (gtf.features_of_type('gene'))
yes = 0
no = 0
for gene in genes:
    transcripts = gtf.children(gene, featuretype="transcript")
    for transcript in transcripts:
        if 'transcript_support_level' in transcript.attributes:
            print(transcript['transcript_biotype'][0])
            if transcript.attributes['transcript_support_level'][0] not in ['1']:
                no += 1
                continue
        else:
            no += 1
            continue
        yes += 1
    
print(yes)
print(no)


TEC
snRNA
protein_coding
processed_transcript
processed_transcript
processed_pseudogene
TEC
TEC
TEC
antisense
TEC
processed_pseudogene
TEC
sense_intronic
TEC
TEC
snRNA
lincRNA
lincRNA
protein_coding
protein_coding
protein_coding
retained_intron
processed_pseudogene
TEC
protein_coding
protein_coding
protein_coding
protein_coding
retained_intron
protein_coding
protein_coding
protein_coding
retained_intron
antisense
processed_pseudogene
snRNA
processed_pseudogene
lincRNA
processed_pseudogene
processed_pseudogene
processed_pseudogene
snRNA
processed_pseudogene
processed_pseudogene
nonsense_mediated_decay
processed_transcript
protein_coding
retained_intron
protein_coding
protein_coding
retained_intron
TEC
protein_coding
protein_coding
protein_coding
protein_coding
nonsense_mediated_decay
protein_coding
retained_intron
protein_coding
nonsense_mediated_decay
protein_coding
protein_coding
processed_pseudogene
TEC
protein_coding
protein_coding
protein_coding
processed_transcript
protein_coding


KeyboardInterrupt: 

In [24]:
genes = (gtf.features_of_type('gene'))

for gene in genes:
    transcripts = gtf.children(gene, featuretype="transcript")
    for transcript in transcripts:
        print(transcript['transcript_support_level'][0])
        if transcript['transcript_support_level'][0] not in ['1']:
            no += 1
            continue
        yes += 1

print(yes)
print(no)

NA
NA
1
1
1
NA
NA
NA
NA
3
NA
NA
NA
1
NA
NA
NA
1
3
1
1
5
1
NA
NA
1
1
5
1
NA
1
5
5
NA
3
NA
NA
NA
1
NA
NA
NA
NA
NA
NA
1
5
1
2
1
1
NA
NA
1
2
3
3
1
5
1
1
5
1
5
NA
NA
1
5
3
3
2
1
5
NA
NA
1
NA
1
3
1
5
5
5
3
1
NA
NA
NA
1
1
5
1
NA
NA
NA
NA
NA
NA
1
3
2
1
1
5
1
2
3
2
5
1
1
5
1
5
NA
3
NA
1
NA
NA
NA
1
3
3
1
3
3
3
1
3
1
5
1
5
NA
NA
NA
NA
NA
NA
NA
1
1
5
2
5
1
1
5
NA
NA
NA
NA
NA
NA
1
3
NA
1
NA
1
NA
NA
NA
NA
1
2
1
5
1
3
1
1
1
1
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
NA
1
3
1
5
1
1
1
3
1
1
NA
1
5
1
5
2
3
5
1
1
1
3
5
3
5
5
1
1
1
1
1
NA
1
1
1
3
3
1
NA
NA
1
5
5
5
5
2
2
2
1
2
2
NA
NA
5
NA
1
5
1
2
1
5
3
NA
1
2
2
2
5
3
NA
1
1
1
1
1
3
3
3
3
3
1
2
5
5
3
2
1
5
5
3
2
NA
NA
3
3
1
3
NA
5
3
5
2
1
5
NA
2
NA
NA
1
2
1
1
NA
NA
NA
NA
NA
NA
NA
NA
1
1
5
3
5
1
1
NA
NA
NA
NA
1
1
1
1
2
1
3
5
1
5
NA
NA
NA
NA
NA
NA
NA
NA
1
NA
1
1
1
1
1
1
1
1
3
3
5
5
3
1
1
1
5
1
NA
NA
5
1
1
5
1
1
2
1
1
3
3
1
NA
NA
NA
NA
NA
NA
NA
NA
NA
3
NA
1
3
3
1
2
NA
1
NA
NA
NA
NA
NA
5
1
1
1
3
3
3
2
1
NA
NA
NA
5
1
NA
NA
3
NA
NA
NA
1
NA
NA


KeyError: 'transcript_support_level'

In [27]:
import numpy as np
import collections
from tqdm import tqdm
import pandas as pd
import io
import os
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import pickle
import re
import sys
import h5py
from math import ceil
from collections import defaultdict
from scipy.sparse import lil_matrix,csr_matrix,coo_matrix,dok_matrix, save_npz
import pyfastx
import gffutils

def create_datapoints(seq, strand, tx_start, tx_end):
    # This function first converts the sequence into an integer array, where
    # A, C, G, T, Missing are mapped to 1, 2, 3, 4, 5 respectively. If the strand is
    # negative, then reverse complementing is done. . It then calls reformat_data and one_hot_encode

    seq = seq.upper()
    seq = re.sub(r'[^AGTC]', '5',seq)
    seq = seq.replace('A', '1').replace('C', '2')
    seq = seq.replace('G', '3').replace('T', '4')

    tx_start = int(tx_start)
    tx_end = int(tx_end) 

    Y_idx = []
    
    X0 = np.asarray([int(x) for x in seq])

    X = one_hot_encode(X0)

    return X

def ceil_div(x, y):
    return int(ceil(float(x)/y))


IN_MAP = np.asarray([[0, 0, 0, 0,0],
                     [1, 0, 0, 0,0],
                     [0, 1, 0, 0,0],
                     [0, 0, 1, 0,0],
                     [0, 0, 0, 1,0],
                    [0, 0, 0, 0,1]])
# One-hot encoding of the inputs: 0 is for padding, and 1, 2, 3, 4 correspond
# to A, C, G, T, Missing respectively.

OUT_MAP = np.asarray([[1, 0, 0],
                      [0, 1, 0],
                      [0, 0, 1],
                      [0, 0, 0]])

def one_hot_encode(Xd):
    return IN_MAP[Xd.astype('int8')]

def getJunctions(gtf,transcript_id):
    transcript = gtf[transcript_id.split('.')[0]]
    strand = transcript[6]
    exon_junctions = []
    tx_start = int(transcript[3])
    tx_end = int(transcript[4])
    exons = gtf.children(transcript, featuretype="exon")
    for exon in exons:
        exon_start = int(exon[3])
        exon_end = int(exon[4])
        exon_junctions.append((exon_start,exon_end))

    intron_junctions = []

    if strand=='+':
        intron_start = exon_junctions[0][1]
        for i,exon_junction in enumerate(exon_junctions[1:]):
            intron_end = exon_junction[0]
            intron_junctions.append((intron_start,intron_end))
            if i+1 != len(exon_junctions[1:]):
                intron_start = exon_junction[1]

    elif strand=='-':
        exon_junctions.reverse()
        intron_start = exon_junctions[0][1]
        for i,exon_junction in enumerate(exon_junctions[1:]):
            intron_end = exon_junction[0]
            intron_junctions.append((intron_start,intron_end))
            if i+1 != len(exon_junctions[1:]):
                intron_start = exon_junction[1]

    jn_start = [x[0] for x in intron_junctions]
    jn_end = [x[1] for x in intron_junctions]
    Y_type, Y_idx = [],[]
    if strand == '+':
        Y0 = -np.ones(tx_end-tx_start+1)
        if len(jn_start) > 0:
            Y0 = np.zeros(tx_end-tx_start+1)
            for c in jn_start:
                if tx_start <= c <= tx_end:
                    Y_type.append(2)
                    Y_idx.append(c-tx_start)
            for c in jn_end:
                if tx_start <= c <= tx_end:
                    Y_type.append(1)
                    Y_idx.append(c-tx_start)

    elif strand == '-':
        Y0 = -np.ones(tx_end-tx_start+1)

        if len(jn_start) > 0:
            Y0 = np.zeros(tx_end-tx_start+1)
            for c in jn_end:
                if tx_start <= c <= tx_end:
                    Y_type.append(2)
                    Y_idx.append(tx_end-c)
            for c in jn_start:
                if tx_start <= c <= tx_end:
                    Y_type.append(1)
                    Y_idx.append(tx_end-c)

    return jn_start,jn_end,Y_type, Y_idx


def createDataset(gtf,fasta,setType,data_dir):
    genes = gtf.features_of_type('gene')

    if setType == 'train':
            CHROM_GROUP = ['chr11', 'chr13', 'chr15', 'chr17', 'chr19', 'chr21',
                           'chr2', 'chr4', 'chr6', 'chr8', 'chr10', 'chr12',
                           'chr14', 'chr16', 'chr18', 'chr20', 'chr22', 'chrX', 'chrY']
    elif setType == 'test':
        CHROM_GROUP = ['chr1', 'chr3', 'chr5', 'chr7', 'chr9']
    else:
        CHROM_GROUP = ['chr1', 'chr3', 'chr5', 'chr7', 'chr9',
                       'chr11', 'chr13', 'chr15', 'chr17', 'chr19', 'chr21',
                       'chr2', 'chr4', 'chr6', 'chr8', 'chr10', 'chr12',
                       'chr14', 'chr16', 'chr18', 'chr20', 'chr22', 'chrX', 'chrY']

    idx = 0
    
    prev_start = None
    prev_end = None
    prev_chrom = None
    
    seqData = {}
    geneToLabel = {}
    transcriptToLabel = {}
    
    for chrom in CHROM_GROUP:
        seqData[chrom] = dok_matrix((len(fasta[chrom]), 5), dtype=np.int8)
    
    if os.path.exists('{}/annotation_ensembl_v87_{}.txt'.format(data_dir,setType)):
        os.remove('{}/annotation_ensembl_v87_{}.txt'.format(data_dir,setType))

    os.makedirs(os.path.join(data_dir, 'sparse_sequence_data'), exist_ok=True)
    
    for gene in tqdm(genes): 
        chrom = 'chr' + gene[0]
        
        if chrom not in CHROM_GROUP:
            continue
        current_chrom = chrom
        if prev_chrom is None:
            prev_chrom = current_chrom
        strand = gene[6]
        gene_start = gene[3]
        gene_end = gene[4]
        transcripts = gtf.children(gene, featuretype="transcript")
        for transcript in transcripts:
            transcript_id = transcript['transcript_id'][0]
            if transcript['transcript_biotype'][0]!='protein_coding':
                continue
            if 'transcript_support_level' in transcript.attributes:
                if transcript.attributes['transcript_support_level'][0] not in ['1']:
                    continue
            else:
                continue

            jn_start,jn_end,Y_type, Y_idx = getJunctions(gtf,transcript_id)
            
            tx_start = int(transcript[3])
            tx_end = int(transcript[4])

            if (gene_start!=prev_start and gene_end!=prev_end):
                try:
                    seq = fasta[chrom][int(gene_start)-1:int(gene_end)]
                    seq = seq.seq
                except:
                    print('Failed reading fasta file for {}:{}-{}'.format(chrom,gene_start,gene_end))
                    print('SKIPPING')
                    break
                X = create_datapoints(seq, strand, gene_start, gene_end)

                seqData[chrom][int(gene_start)-1:int(gene_end)] = X
                prev_start,prev_end = gene_start,gene_end               

            transcriptToLabel[transcript_id] = (Y_type, Y_idx)
            
            if prev_chrom != current_chrom:
                save_npz('{}/sparse_sequence_data/{}_{}.npz'.format(data_dir,prev_chrom,setType), seqData[prev_chrom].tocoo())
                seqData[prev_chrom] = None

            prev_chrom = current_chrom

            name = '{}---{}.{}---{}.{}---{}'.format(gene['gene_name'][0],gene['gene_id'][0],gene['gene_version'][0],transcript['transcript_id'][0],transcript['transcript_version'][0],transcript['transcript_biotype'][0])
            
            if strand=='+':
                with open('{}/annotation_ensembl_v87_{}.txt'.format(data_dir,setType), 'a') as the_file:
                    the_file.write('{}\t{}\t{}\t{}\t{}\t{}\t{}\n'.format(name,chrom,strand,tx_start,tx_end,','.join([str(x) for x in jn_start]),','.join([str(x) for x in jn_end])))
            if strand=='-':
                with open('{}/annotation_ensembl_v87_{}.txt'.format(data_dir,setType), 'a') as the_file:
                    the_file.write('{}\t{}\t{}\t{}\t{}\t{}\t{}\n'.format(name,chrom,strand,tx_start,tx_end,','.join([str(x) for x in jn_end]),','.join([str(x) for x in jn_start])))

                    
    save_npz('{}/sparse_sequence_data/{}_{}.npz'.format(data_dir,prev_chrom,setType), seqData[prev_chrom].tocoo())

    with open('{}/sparse_discrete_label_data_{}.pickle'.format(data_dir,setType), 'wb') as handle:
        pickle.dump(transcriptToLabel, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [28]:
print('Creating test data')
createDataset(gtf,fasta,'test',data_dir)

Creating test data


49671it [39:50, 20.78it/s]   


In [9]:
print(len(fasta.longest))

195154279


In [6]:
def getData(data_dir,setType):
    if setType == 'train':
        chroms = ['chr11', 'chr13', 'chr15', 'chr17', 'chr19', 'chr21',
                               'chr2', 'chr4', 'chr6', 'chr8', 'chr10', 'chr12',
                               'chr14', 'chr16', 'chr18', 'chr20', 'chr22', 'chrX', 'chrY']
    if setType == 'test':
        chroms = ['chr1', 'chr3', 'chr5', 'chr7', 'chr9']
    if setType == 'all':
        chroms = ['chr11', 'chr13', 'chr15', 'chr17', 'chr19','chr1', 'chr21',
                               'chr2', 'chr4', 'chr6', 'chr8', 'chr10', 'chr12',
                               'chr14', 'chr16', 'chr18', 'chr20', 'chr22', 'chr3', 'chr5', 'chr7', 'chr9', 'chrX', 'chrY']
    if setType == 'all':
        with open('{}/sparse_discrete_label_data_train.pickle'.format(data_dir), 'rb') as handle:
            transcriptToLabel = pickle.load(handle)
        with open('{}/sparse_discrete_label_data_test.pickle'.format(data_dir), 'rb') as handle:
            transcriptToLabel2 = pickle.load(handle)
            transcriptToLabel.update(transcriptToLabel2)
    else:
        with open('{}/sparse_discrete_label_data_{}.pickle'.format(data_dir,setType), 'rb') as handle:
            transcriptToLabel = pickle.load(handle)
    
    if setType == 'all':
        annotation1 = pd.read_csv(data_dir+'/annotation_ensembl_v87_train.txt',sep='\t',header=None)[[0,1,2,3,4]]
        annotation2 = pd.read_csv(data_dir+'/annotation_ensembl_v87_test.txt',sep='\t',header=None)[[0,1,2,3,4]]
        annotation = pd.concat([annotation1,annotation2],axis=0)
    else:
        annotation = pd.read_csv(data_dir+'/annotation_ensembl_v87_{}.txt'.format(setType),sep='\t',header=None)[[0,1,2,3,4]]
    annotation.columns = ['name','chrom','strand','tx_start','tx_end']
    annotation['transcript'] = annotation['name'].apply(lambda x: x.split('---')[-2].split('.')[0]).values
    annotation['gene'] = annotation['name'].apply(lambda x: x.split('---')[-3].split('.')[0]).values
    #annotation['support'] = annotation['transcript'].apply(lambda x:transcriptToSupport[x])

    chrom_paths = glob(data_dir+'/sparse_sequence_data/*')
    chromToPath = {}
    for path in chrom_paths:
        chromToPath[path.split('/')[-1].split('_')[0]] = path

    seqData = {}
    for chrom in chroms:
        path = glob(data_dir+'/sparse_sequence_data/{}_*.npz'.format(chrom))[0]
        seqData[chrom] = load_npz(path).tocsr()
    return annotation,transcriptToLabel,seqData

In [17]:
data_dir = '../Data/'
fasta_file_path = '../Data/hg38.fa'
gtf_file_path = '../Data/Homo_sapiens.GRCh38.87.gtf'
fasta = pyfastx.Fasta(fasta_file_path)
fname = data_dir+gtf_file_path.split('/')[-1][:-4]+'.db'
if not os.path.isfile(fname): 
    gffutils.create_db(gtf_file_path, fname, force=True, disable_infer_genes=True, disable_infer_transcripts=True)

gtf = gffutils.FeatureDB(fname)

In [75]:
genes = gtf.features_of_type('gene')
seqData = {}
for chrom in ['chr1', 'chr3', 'chr5', 'chr7', 'chr9']:
    seqData[chrom] = dok_matrix((len(fasta[chrom]), 5), dtype=np.int8)


def create_datapoints(seq, strand, tx_start, tx_end):
    # This function first converts the sequence into an integer array, where
    # A, C, G, T, Missing are mapped to 1, 2, 3, 4, 5 respectively. If the strand is
    # negative, then reverse complementing is done. . It then calls reformat_data and one_hot_encode

    seq = seq.upper()
    print(len(seq))
    seq = re.sub(r'[^AGTC]', '5',seq)
    seq = seq.replace('A', '1').replace('C', '2')
    seq = seq.replace('G', '3').replace('T', '4')

    tx_start = int(tx_start)
    tx_end = int(tx_end) 

    Y_idx = []
    
    X0 = np.asarray([int(x) for x in seq])

    X = one_hot_encode(X0)

    return X


for gene in genes:
    gene_start = gene[3]
    gene_end = gene[4]
    chrom = 'chr' + gene[0]
    strand = gene[6]
    seq = fasta[chrom][int(gene_start)-1:int(gene_end)]
    seq = seq.seq
    if len(seq) > 45000:
        X = create_datapoints(seq, strand, gene_start, gene_end)
        with np.printoptions(threshold=np.inf):
            print(np.shape(X))
        break

136229
(136229, 5)


In [7]:
annotation, transcriptToLabel, seqData = getData('../Data', 'test')

In [29]:
def getDataPointListFull(annotation,transcriptToLabel,SL,CL_max,shift,include_pos=False):
    data = []
    for idx in range(annotation.shape[0]):
        transcript,gene,chrom,strand,tx_start,tx_end = annotation['transcript'].values[idx], annotation['gene'].values[idx],annotation['chrom'].values[idx],annotation['strand'].values[idx],annotation['tx_start'].values[idx],annotation['tx_end'].values[idx]
        length = tx_end-tx_start+1
        num_points = ceil_div(length, shift)
        label = [np.array(x) for x in transcriptToLabel[transcript]]
        for i in range(num_points):
            if strand=='+':
                start,end = tx_start+shift*i,tx_start+SL+shift*(i)-1
                if i == 0:
                    start_point = start
                inRange = [l>=start-start_point-CL_max//2 and l<=end-start_point+CL_max//2 for l in label[1]]
                mask_l = tx_start-np.min([start-CL_max//2,tx_start])
                mask_r = np.max([end+CL_max//2,tx_end])-tx_end
                print('chrom:',chrom)
                print('Start:', start)
                print('End:', end)
                print('mask_l:', mask_l)
                print('mask_r', mask_r)
                print('label[1][inRange]',label[1][inRange])
                print('label[0][inRange]',label[0][inRange])
                print((end+CL_max//2-mask_r)-(start-CL_max//2-1+mask_l))
                print('\n')
                data.append(DataPointFull(transcript,gene,chrom,strand,start,end,tx_start,tx_end,label[1][inRange],label[0][inRange],SL,CL_max,start-start_point,mask_l,mask_r,include_pos))
            else:
                start,end = tx_end-SL-shift*i+1,tx_end-shift*i
                if i == 0:
                    start_point = end
                inRange = [l>=start_point-end-CL_max//2 and l<= start_point-start+CL_max//2 for l in label[1]]
                mask_l =  np.max([end+CL_max//2,tx_end])-tx_end
                mask_r = tx_start-np.min([start-CL_max//2,tx_start])
                data.append(DataPointFull(transcript,gene,chrom,strand,start,end,tx_start,tx_end,label[1][inRange],label[0][inRange],SL,CL_max,start_point-end,mask_l,mask_r,include_pos))
    return data

In [30]:
SL = 5000
CL_max = 40000
data = spliceDataset(getDataPointListFull(annotation,transcriptToLabel,SL,CL_max,shift=SL))

chrom: chr1
Start: 182393
End: 187392
mask_l: 20000
mask_r 23234
label[1][inRange] [ 353  847  721 1529]
label[0][inRange] [2 2 1 1]
1766


chrom: chr1
Start: 960587
End: 965586
mask_l: 20000
mask_r 19871
label[1][inRange] [ 213  965 1163 1460 1884 2330 2666 2917 3421 3593 3943  706 1042 1239
 1768 2117 2522 2750 3333 3520 3762 4376]
label[0][inRange] [2 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1]
5129


chrom: chr1
Start: 965587
End: 970586
mask_l: 15000
mask_r 24871
label[1][inRange] [ 213  965 1163 1460 1884 2330 2666 2917 3421 3593 3943  706 1042 1239
 1768 2117 2522 2750 3333 3520 3762 4376]
label[0][inRange] [2 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1]
5129


chrom: chr1
Start: 966502
End: 971501
mask_l: 20000
mask_r 16493
label[1][inRange] [ 112  301 3921 4099 4256 4504 4706 4902 5648 5922 6508 7138 7549 7862
  202 3775 4019 4184 4377 4575 4822 5573 5786 6359 6998 7331 7814 7940]
label[0][inRange] [2 2 2 2 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
8507


chrom: chr1
Start

KeyboardInterrupt: 

In [9]:
from tqdm import tqdm

data.seqData = seqData

for input, output in tqdm(data):
    break

  0%|          | 0/133024 [00:00<?, ?it/s]


In [28]:
start= 183497527
end=183502526
mask_l=0
mask_r=0
print(start-CL_max//2-1+mask_l)
print(end+CL_max//2-mask_r)
with np.printoptions(threshold=np.inf):
    seq = seqData['chr1'][start-CL_max//2-1+mask_l:end+CL_max//2-mask_r,:4].toarray().T
    print(((seq)))


183477526
183522526
[[0 1 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1
  0 0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0
  0 1 0 1 0 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1
  0 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1 0 1 0 0
  0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 1 0 1
  0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 0 1 0 1 0 1
  0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 1
  0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 1 0 1 0 1 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0
  0 1 0 1 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 1 0
  0 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 1 0 1 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0
  0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 1 0 0 1 1 0 1 0 1 1 0 1 0 0 0
  1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 1 0 1 1 1 1 1 0 0 0 0 0 0
  0 0 1 0 0 0 1 1 1 0 1 0 0 1 1 0 0 1 1 1 1 0 1 0 0 0 1 1 0 1 0 0 0 0 1 0
  1 1 1 1 0 1 1 1 